In [5]:
import pandas as pd
import numpy as np
import os
import glob

import pyarrow as pa
import pyarrow.parquet as pq

In [6]:
import requests
current_date = "2026-01"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"

end_use = "naics?get=ALL_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/exports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url)

df = pd.DataFrame(r.json()[1:])
df.columns = r.json()[0]
df["total_exports"] = df["ALL_VAL_MO"].astype(float)
df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])
top_products = grp.agg({"total_exports": "sum", "CTY_CODE": "first"})

country_list = list(top_products.sort_values(by="total_exports", ascending=False).CTY_CODE)[0:31]
country_list[0] = ""
country_list.extend(["0003", "0020"])

In [7]:
# data_dir = ".\\data\\exports-hs10"
# only run if in the folder locally

# for xxx in country_list:
    
#     country_label = "TOTAL" if xxx == "" else xxx
    
#     # Find all monthly files for this country
#     pattern = os.path.join(data_dir, f"{country_label}data-*.parquet")
#     monthly_files = sorted(glob.glob(pattern))
    
#     # Exclude any existing combined file from the list
#     combined_file = os.path.join(data_dir, f"{country_label}data-combined.parquet")
#     monthly_files = [f for f in monthly_files if "combined" not in f]
    
#     if not monthly_files:
#         print(f"No files found for {country_label}, skipping...")
#         continue
    
#     print(f"Combining {len(monthly_files)} monthly files for {country_label}...")
    
#     dfs = []
#     for f in monthly_files:
#         try:
#             dfs.append(pq.read_table(f).to_pandas())
#         except Exception as e:
#             print(f"  Error reading {f}: {e}")
    
#     if not dfs:
#         print(f"  No data loaded for {country_label}, skipping...")
#         continue
    
#     combined = pd.concat(dfs, ignore_index=True)
    
#     pq.write_table(pa.Table.from_pandas(combined), combined_file)
    
#     print(f"  Saved {len(combined):,} rows to {combined_file}")

# print("Done!")

In [8]:
# Append new monthly files to existing combined files
data_dir = ".\\data\\exports-hs10"
new_month = current_date  # e.g. "2026-01"

for xxx in country_list:
    country_label = "TOTAL" if xxx == "" else xxx
    combined_file = os.path.join(data_dir, f"{country_label}data-combined.parquet")
    new_file = os.path.join(data_dir, f"{country_label}data-{new_month}.parquet")

    if not os.path.exists(new_file):
        print(f"No new file for {country_label}, skipping...")
        continue

    if not os.path.exists(combined_file):
        print(f"No existing combined file for {country_label}, skipping...")
        continue

    existing = pq.read_table(combined_file).to_pandas()
    new_data = pq.read_table(new_file).to_pandas()
    updated = pd.concat([existing, new_data], ignore_index=True)
    pq.write_table(pa.Table.from_pandas(updated), combined_file)
    print(f"Updated {country_label}: added {len(new_data):,} rows (total: {len(updated):,})")

print("Done!")

No existing combined file for TOTAL, skipping...
Updated 1220: added 5,992 rows (total: 1,022,333)
Updated 2010: added 6,369 rows (total: 1,213,579)
Updated 5700: added 3,762 rows (total: 923,586)
Updated 5880: added 3,809 rows (total: 912,084)
Updated 4120: added 3,920 rows (total: 927,401)
Updated 4280: added 3,716 rows (total: 870,706)
Updated 4210: added 3,061 rows (total: 766,064)
Updated 5800: added 3,352 rows (total: 844,869)
Updated 3510: added 3,018 rows (total: 746,937)
Updated 4279: added 2,811 rows (total: 726,230)
Updated 5590: added 2,904 rows (total: 725,992)
Updated 5830: added 2,568 rows (total: 704,464)
Updated 4231: added 2,298 rows (total: 618,995)
Updated 5820: added 2,383 rows (total: 704,534)
Updated 5330: added 2,937 rows (total: 714,484)
Updated 4419: added 1,648 rows (total: 503,107)
Updated 6021: added 3,392 rows (total: 843,522)
Updated 4759: added 2,659 rows (total: 711,566)
Updated 5200: added 2,595 rows (total: 677,296)
Updated 3010: added 2,725 rows (tot

In [9]:
data_dir = ".\\data\\exports-hs10"

# Find all country-level combined files, excluding TOTAL and USMCA (0020)
exclude_labels = {"TOTAL", "0020"}

all_combined = sorted(glob.glob(os.path.join(data_dir, "*data-combined.parquet")))
country_files = [
    f for f in all_combined
    if not any(os.path.basename(f).startswith(lbl + "data-") for lbl in exclude_labels)
]

print(f"Combining {len(country_files)} country files into All-country-exports.parquet...")

dfs = []
for f in country_files:
    try:
        dfs.append(pq.read_table(f).to_pandas())
    except Exception as e:
        print(f"  Error reading {f}: {e}")

if dfs:
    all_countries = pd.concat(dfs, ignore_index=True)
    out_path = os.path.join(data_dir, "All-country-exports.parquet")
    pq.write_table(pa.Table.from_pandas(all_countries), out_path)
    print(f"Saved {len(all_countries):,} rows to {out_path}")
else:
    print("No data loaded, nothing saved.")

Combining 31 country files into All-country-exports.parquet...
Saved 22,776,430 rows to .\data\exports-hs10\All-country-exports.parquet
